# Adaptive-q MFA evaluation

This notebook evaluates an ARD or HDDC adaptive-q model trained on the toy-manifold activation shards. It contains only:

- train and validation NLL,
- cluster homogeneity, completeness, ARI, and NMI,
- learned rank of $W_k$ versus the planted manifold rank (exact and within one), and
- the number of dead components (zero assigned points).

Set `MODEL_DIR` below to the completed run directory. The first executable check requires the standard `mfa_model_assignments.pt` artifact and does not generate it.

In [1]:
from pathlib import Path
import os
import sys

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = REPO.parent.resolve()
if not (REPO / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks/.")

MODEL_DIR = Path(
   "/u/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/toy_manifold_models_20k/adaptive_q_toy_20k_ard/ard__toy_manifolds_d128_20k_noise1e4__l00__k200__q16__s42__4c877aa2"
)
MODEL_PATH = MODEL_DIR / "mfa_model.pt"
ASSIGNMENTS_PATH = MODEL_DIR / "mfa_model_assignments.pt"

if not ASSIGNMENTS_PATH.is_file():
    raise FileNotFoundError(
        f"Required MFA assignment artifact is missing: {ASSIGNMENTS_PATH}\n"
        "Create the full mfa_model_assignments.pt artifact in the model folder before evaluation."
    )

sys.path.insert(0, str(REPO / "src"))
print(f"Using assignment artifact: {ASSIGNMENTS_PATH}")

Using assignment artifact: /u/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/toy_manifold_models_20k/adaptive_q_toy_20k_ard/ard__toy_manifolds_d128_20k_noise1e4__l00__k200__q16__s42__4c877aa2/mfa_model_assignments.pt


## Load and validate the run

The assignments must cover the complete canonical toy-manifold stream. The saved validation rows define the validation split; every other row is part of the training split.

In [2]:
import json

import pandas as pd
import torch
from torch.utils.data import DataLoader

from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EVAL_BATCH_SIZE = 4096

required_paths = [MODEL_DIR / "config.json", MODEL_DIR / "val_indices.json"]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing run artifacts: {missing}")

run_config = json.loads((MODEL_DIR / "config.json").read_text())
split_info = json.loads((MODEL_DIR / "val_indices.json").read_text())
model_kind = run_config.get("model")
has_model_artifact = MODEL_PATH.is_file() or (
    model_kind == "MFA_HDDC" and (MODEL_DIR / "mfa_model_shards.json").is_file()
)
if not has_model_artifact:
    raise FileNotFoundError(
        f"Missing model artifact: expected {MODEL_PATH}"
        + (" or mfa_model_shards.json" if model_kind == "MFA_HDDC" else "")
    )

if model_kind == "MFA_ARD":
    from dalg.models.adaptive_q.mfa_ard import load_mfa_ard

    model = load_mfa_ard(MODEL_PATH, map_location="cpu")
elif model_kind == "MFA_HDDC":
    from dalg.models.adaptive_q.mfa_hddc import load_mfa_hddc

    model = load_mfa_hddc(MODEL_PATH, map_location="cpu")
else:
    raise ValueError(
        f"Expected config model to be 'MFA_ARD' or 'MFA_HDDC', got {model_kind!r}."
    )

shard_dir = Path(run_config["shard_dir"]).expanduser()
if not shard_dir.is_absolute():
    shard_dir = REPO / shard_dir
shard_config = json.loads((shard_dir / "config.json").read_text())
if shard_config.get("source_kind") != "toy_manifolds":
    raise ValueError("Rank recovery requires shards produced by save_toy_manifold_shards.")
if int(shard_config["window"]) != 1 or int(shard_config.get("drop_prefix", 0)) != 0:
    raise ValueError("Expected one activation per toy-manifold row.")

metadata_path = shard_dir / shard_config["manifold_metadata"]
manifold_metadata = torch.load(metadata_path, map_location="cpu", weights_only=True)
row_manifold_ids = manifold_metadata["row_manifold_ids"].reshape(-1).long()

assignment_bundle = torch.load(
    ASSIGNMENTS_PATH, map_location="cpu", mmap=True, weights_only=True
)
assignments = assignment_bundle["assignments"].reshape(-1).long()
cluster_sizes = assignment_bundle["cluster_sizes"].reshape(-1).long()

layer = int(run_config["layer"])
meta_index = load_meta_index(shard_dir, layer=layer)
if assignments.numel() != len(meta_index) or assignments.numel() != row_manifold_ids.numel():
    raise ValueError(
        "Assignments must cover the complete canonical toy-manifold stream: "
        f"assignments={assignments.numel()}, metadata rows={len(meta_index)}, "
        f"manifold labels={row_manifold_ids.numel()}."
    )
if cluster_sizes.numel() != model.K or int(assignment_bundle["K"]) != model.K:
    raise ValueError("Assignment K does not match the loaded model.")
if not torch.equal(torch.bincount(assignments, minlength=model.K), cluster_sizes):
    raise ValueError("cluster_sizes is inconsistent with assignments.")

val_global_rows = set(split_info["val_global_rows"])
val_positions = [
    position for position, row in enumerate(meta_index)
    if row["global_row"] in val_global_rows
]
val_position_set = set(val_positions)
train_positions = [
    position for position in range(len(meta_index))
    if position not in val_position_set
]
if len(train_positions) != int(split_info["train_rows"]):
    raise ValueError("Reconstructed training split does not match val_indices.json.")
if len(val_positions) != int(split_info["val_rows"]):
    raise ValueError("Reconstructed validation split does not match val_indices.json.")

model = model.to(DEVICE).eval()
print(
    f"Loaded {model_kind}: K={model.K}, D={model.D}, q_max={model.q}, "
    f"train rows={len(train_positions):,}, val rows={len(val_positions):,}, device={DEVICE}."
)

Loaded MFA_ARD: K=200, D=128, q_max=16, train rows=18,000, val rows=2,000, device=cuda.


## 1. Train and validation NLL

Both values are recomputed from `mfa_model.pt` over the exact saved split. For ARD, this is pure MFA NLL and excludes the ARD parameter penalty.

In [3]:
@torch.no_grad()
def evaluate_nll(row_positions):
    dataset = ActivationBatchDataset(
        shard_dir,
        layer=layer,
        row_subset=row_positions,
        batch_size=EVAL_BATCH_SIZE,
        drop_prefix=0,
        dtype=torch.float32,
        shuffle_shards=False,
        shuffle_within_shard=False,
        seed=0,
    )
    loader = DataLoader(dataset, batch_size=None, num_workers=0)
    total_nll = 0.0
    total_points = 0
    with model.inference_cache():
        for batch in loader:
            x = batch.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            total_nll += float(model.nll(x).item()) * x.shape[0]
            total_points += x.shape[0]
    if total_points == 0:
        raise ValueError("Cannot evaluate NLL on an empty split.")
    return total_nll / total_points

nll_results = pd.DataFrame(
    {"NLL": [evaluate_nll(train_positions), evaluate_nll(val_positions)]},
    index=["train", "validation"],
)
display(nll_results)

,NLL
train,-431.780846
validation,-431.771174


## 2. Cluster homogeneity and completeness

Ground-truth classes are manifold-instance IDs and predicted clusters are MFA responsibility-argmax assignments.

In [4]:
from sklearn.metrics import (
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
)

clustering_results = pd.DataFrame(
    {
        "value": [
            homogeneity_score(row_manifold_ids.numpy(), assignments.numpy()),
            completeness_score(row_manifold_ids.numpy(), assignments.numpy()),
            adjusted_rand_score(row_manifold_ids.numpy(), assignments.numpy()),
            normalized_mutual_info_score(row_manifold_ids.numpy(), assignments.numpy()),
        ]
    },
    index=["homogeneity", "completeness", "adjusted Rand index", "normalized mutual information"],
)
display(clustering_results)

,value
homogeneity,1.000000
completeness,0.154711
adjusted Rand index,0.025354
normalized mutual information,0.267965


## 3. Rank of $W_k$ versus manifold rank

Each non-empty component is paired with the manifold instance contributing the most assigned points. ARD rank is `effective_ranks()`; HDDC rank is `component_ranks`. The comparison excludes dead components because they have no associated ground-truth manifold.

In [5]:
if model_kind == "MFA_ARD":
    w_ranks = model.effective_ranks().cpu().long()
else:
    w_ranks = model.component_ranks.cpu().long()

num_manifolds = int(manifold_metadata["num_manifolds"])
component_by_manifold = torch.bincount(
    assignments * num_manifolds + row_manifold_ids,
    minlength=model.K * num_manifolds,
).reshape(model.K, num_manifolds)
dominant_manifold = component_by_manifold.argmax(dim=1)
manifold_ranks = torch.tensor(
    [int(manifold["intrinsic_dim"]) for manifold in manifold_metadata["manifolds"]],
    dtype=torch.long,
)
live_components = cluster_sizes > 0
live_ids = torch.where(live_components)[0]
matched_manifold_ranks = manifold_ranks[dominant_manifold[live_components]]
live_w_ranks = w_ranks[live_components]

rank_comparison = pd.DataFrame(
    {
        "component": live_ids.numpy(),
        "W_rank": live_w_ranks.numpy(),
        "manifold_id": dominant_manifold[live_components].numpy(),
        "manifold_rank": matched_manifold_ranks.numpy(),
        "rank_error": (live_w_ranks - matched_manifold_ranks).numpy(),
    }
)
rank_summary = pd.DataFrame(
    {
        "value": [
            float(live_w_ranks.float().mean()),
            float((live_w_ranks == matched_manifold_ranks).float().mean()),
            float(((live_w_ranks - matched_manifold_ranks).abs() <= 1).float().mean()),
            float((live_w_ranks - matched_manifold_ranks).abs().float().mean()),
        ]
    },
    index=["mean learned rank (live)", "exact rank match", "within-one rank match", "mean absolute rank error"],
)
rank_counts = pd.crosstab(
    rank_comparison["manifold_rank"],
    rank_comparison["W_rank"],
    rownames=["manifold rank"],
    colnames=["W rank"],
)
display(rank_summary)
display(rank_counts)
rank_comparison.head()

,value
mean learned rank (live),15.921569
exact rank match,0.000000
within-one rank match,0.000000
mean absolute rank error,14.921569


W rank,8,16
manifold rank,,
1,1,101


,component,W_rank,manifold_id,manifold_rank,rank_error
0,1,16,0,1,15
1,3,16,1,1,15
2,4,16,1,1,15
3,6,16,0,1,15
4,7,16,1,1,15


In [6]:
rank_comparison[rank_comparison["manifold_id"]==0]

,component,W_rank,manifold_id,manifold_rank,rank_error
0,1,16,0,1,15
3,6,16,0,1,15
6,11,8,0,1,7
7,12,16,0,1,15
8,15,16,0,1,15
12,21,16,0,1,15
13,22,16,0,1,15
14,23,16,0,1,15
16,26,16,0,1,15
17,27,16,0,1,15


In [7]:
manifold_metadata

{'config': {'ambient_dim': 128,
  'n_train': 16000,
  'n_val': 4000,
  'calibration_size': 50000,
  'manifolds_per_type': 1,
  'manifold_types': ('circle', 'helix'),
  'offset_radius': 4.0,
  'noise_ratio': 10000.0,
  'seed': 0,
  'segment_min': -1.0,
  'segment_max': 1.0,
  'torus_major_radius': 2.0,
  'torus_minor_radius': 1.0,
  'mobius_half_width': 0.5,
  'swiss_theta_min': 4.71238898038469,
  'swiss_theta_max': 14.137166941154069,
  'swiss_height_min': 0.0,
  'swiss_height_max': 21.0,
  'helix_theta_min': 0.0,
  'helix_theta_max': 12.566370614359172,
  'helix_alpha': 0.2},
 'num_manifolds': 2,
 'manifold_types': ('circle', 'helix'),
 'type_id_to_name': {0: 'circle', 1: 'helix'},
 'type_name_to_id': {'circle': 0, 'helix': 1},
 'intrinsic_dims': (1, 1),
 'embedding_dims': (2, 3),
 'manifold_type_ids': tensor([0, 1]),
 'calibration_means': (tensor([ 0.0014, -0.0005], dtype=torch.float64),
  tensor([-1.3092e-03, -3.6371e-04,  1.2503e+00], dtype=torch.float64)),
 'calibration_scales': 

## 4. Number of dead components

A component is dead when its saved assignment count is zero. For HDDC, skipped low-count components can retain rank `q_max`, so rank summaries should use assignment-live components.

In [8]:
num_live_components = int((cluster_sizes > 0).sum())
num_dead_components = int((cluster_sizes == 0).sum())
pd.DataFrame(
    {"value": [num_live_components, num_dead_components]},
    index=["live components", "dead components"],
)

,value
live components,102
dead components,98


## 5. Summary

One compact table for comparing repeated runs of this notebook. NLL is evaluated on the saved split; clustering and rank metrics use the complete assignment stream.

In [9]:
evaluation_summary = pd.DataFrame(
    {
        "value": [
            nll_results.loc["train", "NLL"],
            nll_results.loc["validation", "NLL"],
            clustering_results.loc["homogeneity", "value"],
            clustering_results.loc["completeness", "value"],
            clustering_results.loc["adjusted Rand index", "value"],
            clustering_results.loc["normalized mutual information", "value"],
            num_live_components,
            num_dead_components,
            rank_summary.loc["mean learned rank (live)", "value"],
            rank_summary.loc["exact rank match", "value"],
            rank_summary.loc["within-one rank match", "value"],
            rank_summary.loc["mean absolute rank error", "value"],
        ]
    },
    index=[
        "train NLL",
        "validation NLL",
        "homogeneity",
        "completeness",
        "adjusted Rand index",
        "normalized mutual information",
        "live components",
        "dead components",
        "mean learned rank (live)",
        "exact rank match",
        "within-one rank match",
        "mean absolute rank error",
    ],
)
evaluation_summary

,value
train NLL,-431.780846
validation NLL,-431.771174
homogeneity,1.000000
completeness,0.154711
adjusted Rand index,0.025354
normalized mutual information,0.267965
live components,102.000000
dead components,98.000000
mean learned rank (live),15.921569
exact rank match,0.000000
